# 20 — Stage B: public judge data — CUDA (Kaggle 2×T4)

Continues jev-lite from the final Stage A model on the Stage B mix (`DATASETS.md` section 8, todo Phase 4b): judge data from public sources plus a replay of Stage A, 1024 tokens, 2 epochs. Evaluates on the mix's `test_in` / `test_ood` (the new sources and the Stage A test sets, whole) and saves the final model. All logic is in `jevlite.train`; this notebook only differs from the Stage A one in the parameters and the train command.

- **Inputs:** Add Input → Notebook Output → the version of `06_build_stage_b` that printed `MIX OK` (the mix is in its `stage_b/`). The `jevlite-mix` dataset is not needed.
- **Start:** `model.init: checkpoint` loads the encoder and the heads of `stage_a-full-s42/final` from the private Hub repo (secret `HF_TOKEN`).
- **Resume:** checkpoints every 40 min to the Hub; if the session ends (12 h), run the notebook again with the same `RUN`. Expect more than one session: ~300k records at up to 1024 tokens.
- **Kaggle settings:** Accelerator GPU T4 x2, Internet On, secret `HF_TOKEN` attached.
- **Heads:** trunk + heads train at `LR_HEADS` (1e-3) in their own group; the log shows `heads <lr>` and `head drift` (relative change of the head weights since the session start). In the smoke run the drift must grow clearly above 0 (Stage A's heads barely moved).
- **Smoke run first:** `MAX_STEPS = 200` checks memory at 1024 tokens (batch 8 per GPU), speed, the eval and a Hub upload. Then `None` and Save & Run All.

## 0. Install (Kaggle / Colab)

In [ ]:
# Kaggle / Colab: install the code from GitHub at a fixed commit. A no-op on a local checkout.
import os, sys

COMMIT = "main"   # set a commit hash for a reproducible run
ON_CLOUD = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or "COLAB_RELEASE_TAG" in os.environ
if ON_CLOUD:
    !rm -rf /tmp/JEV_Judge && git clone -q https://github.com/Nefelim584/JEV_Judge.git /tmp/JEV_Judge
    !cd /tmp/JEV_Judge && git checkout -q {COMMIT} && git log -1 --format='%h %ad %s' --date=short
    !pip install -q --no-deps --ignore-requires-python -e /tmp/JEV_Judge
    !pip install -q loguru peft bitsandbytes
    !pip uninstall -y -q torchao  # the image has torchao 0.10; recent peft refuses < 0.16 instead of skipping it
    sys.path.insert(0, "/tmp/JEV_Judge/src")  # the running kernel does not pick up a fresh editable install
!python --version

## 1. Parameters

The only stack-specific cell.

In [ ]:
STACK = "cuda"
RUN = "stage_b-full-s42"     # one folder locally and on the Hub; the same name resumes the run
STAGE = "stage_b"
INIT = "hub:Nefelim584/jevlite-runs/stage_a-full-s42/final"   # the model Stage B starts from
LR_HEADS = 1e-3              # trunk + heads in their own AdamW group (encoder stays at 3e-5): in Stage A, at 3e-5,
                             # the heads barely left their init (todo, "Stage A — head learning rate")
BATCH = 8                    # per GPU at 1024 tokens (Stage A: 16 at 512); effective batch stays 32 by accumulation
DATA_ROOT = None             # folder holding stage_b/; None = find it among the inputs
RUNS_DIR = "/tmp/jevlite-runs"   # checkpoints: not part of the notebook output (the Hub keeps them)
OUT = "/kaggle/working"          # small outputs to download: logs, eval reports
MAX_MINUTES = 690            # stop with a checkpoint before Kaggle's 12 h limit; re-run to resume
MAX_STEPS = None             # e.g. 200 for a smoke run (ends with an eval and a checkpoint)
HUB = True                   # mirror checkpoints to train.checkpoint.hub_repo (needs HF_TOKEN)
MIX = "stage_b"

## 2. Environment: HF token, devices, paths

In [ ]:
import os
from pathlib import Path

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ and HUB:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF token:", "set" if os.environ.get("HF_TOKEN") else "missing (hub uploads will fail)" if HUB else "not needed")

import torch
import jevlite

REPO = Path(jevlite.__file__).resolve().parents[2]
if DATA_ROOT is None:
    import glob
    hits = glob.glob(f"/kaggle/input/**/{MIX}/train.e0.jsonl", recursive=True)
    assert hits, "add the output of 06_build_stage_b (the version that printed MIX OK) as an input"
    DATA_ROOT = str(Path(hits[0]).parents[1])
NPROC = max(torch.cuda.device_count(), 1)
print("repo:", REPO, "| processes:", NPROC, "| data:", sorted(p.name for p in (Path(DATA_ROOT) / MIX).glob("*.jsonl")))

## 3. Train

One process per GPU (`torchrun`). Logs stream here and go to `RUNS_DIR/RUN/run.log`.

In [ ]:
args = [
    f"--stack {STACK}", f"--stage {STAGE}", f"--run {RUN}", f"--data-root {DATA_ROOT}", f"--runs-dir {RUNS_DIR}",
    "--set model.init=checkpoint", f"--set model.init_checkpoint={INIT}",
    f"--set train.per_device_batch_size={BATCH}", f"--set train.lr.heads={LR_HEADS}",
]
if MAX_MINUTES:
    args.append(f"--max-minutes {MAX_MINUTES}")
if MAX_STEPS:
    args.append(f"--max-steps {MAX_STEPS}")
if not HUB:
    args.append("--no-hub")
cmd = f"torchrun --standalone --nproc_per_node {NPROC} {REPO}/scripts/train.py " + " ".join(args)
print(cmd)
!{cmd}

## 4. Reports

Training curves always; the full eval report only once the run is done (`preds.jsonl` exists). Copied to `OUT` for download.

In [ ]:
import shutil

run_dir = Path(RUNS_DIR) / RUN
out = Path(OUT) / RUN
out.mkdir(parents=True, exist_ok=True)
for name in ("metrics.jsonl", "run.log", "config.yaml"):
    if (run_dir / name).exists():
        shutil.copy(run_dir / name, out / name)
!python {REPO}/scripts/make_report.py training --run {RUN}={run_dir} --out {out}/training
if (run_dir / "preds.jsonl").exists():
    data = f"{DATA_ROOT}/{MIX}/test_in.jsonl {DATA_ROOT}/{MIX}/test_ood.jsonl"
    !python {REPO}/scripts/eval.py --data {data} --pred {run_dir}/preds.jsonl --out {out}/eval --bootstrap 1000
    print((out / "eval" / "report.md").read_text()[:3000])
else:
    print("run not finished: re-run the notebook with the same RUN to resume")